# Task 3 — SmallCNN E3 Experiments

This notebook trains only the two predeclared E3 children. It does **not** retrain E1 or E2.

1. Gender starts from the accepted E1 baseline and changes only to capped class-balanced cross-entropy.
2. Usage starts from the accepted class-balanced E2 model and adds only dropout `p=0.20` between global pooling and the final classifier.

Both models remain the same scratch SmallCNN. Select a Colab GPU runtime, then use Run All.

## 1. Mount Drive and load the submitted branch

Drive supplies the dataset, saved E1/E2 parents, registry, and persistent E3 output.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip().splitlines()
    if dirty:
        print("Local repository changes found:")
        for change in dirty:
            print(f"  {change}")
        print("Trying a safe fast-forward update. Git will stop before overwriting a local file.")
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"Repository ready: {REPO_DIR}")
print(f"Branch: {BRANCH}")
print(f"Commit: {commit}")

## 2. Copy the teacher data onto the runtime disk

Training reads images from Colab's local disk. The archive keeps the repository folder structure.

In [ ]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATA_ZIP}")

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_suffixes = {".jpg", ".jpeg"}

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe_names = [name for name in names if Path(name).is_absolute() or ".." in Path(name).parts]
    if unsafe_names:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive has no teacher images in the expected folder.")
    current_images = sum(path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*"))
    needs_extract = current_images != expected_images or not all(path.is_file() for path in required_files)
    if needs_extract:
        print(f"Extracting {expected_images:,} teacher images...", flush=True)
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*"))
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found {actual_images:,}; "
        f"missing files: {missing_files}"
    )
print(f"Teacher data ready: {actual_images:,} images")

## 3. Resolve the saved parents and verify both E3 children

Gender resolves its five E1 baseline folds. Usage resolves its five accepted E2 class-balanced folds. The checks do not take an optimiser step.

In [ ]:
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

for output_dir in (DRIVE_TASK_DIR / "experiments", DRIVE_TASK_DIR / "logs", DRIVE_TASK_DIR / "results"):
    output_dir.mkdir(parents=True, exist_ok=True)

from fashion.train.task3_experiments import (
    check_task3_child_setup,
    latest_completed_baseline_parent_run_ids,
    latest_completed_usage_e2_parent_run_ids,
)

gender_parent_run_ids = latest_completed_baseline_parent_run_ids(
    "gender", output_root=DRIVE_TASK_DIR
)
usage_parent_run_ids = latest_completed_usage_e2_parent_run_ids(
    output_root=DRIVE_TASK_DIR
)
gender_e3_check = check_task3_child_setup(
    "gender_class_balanced", parent_run_ids=gender_parent_run_ids, root=REPO_DIR, device_name="cuda"
)
usage_e3_check = check_task3_child_setup(
    "usage_classifier_dropout", parent_run_ids=usage_parent_run_ids, root=REPO_DIR, device_name="cuda"
)

print("GPU:", gender_e3_check["environment"]["gpu"])
print("Gender E1 parents:", gender_parent_run_ids)
print("Usage E2 parents: ", usage_parent_run_ids)
print("Gender E3 change:", gender_e3_check["changed_factor"])
print("Usage E3 change: ", usage_e3_check["changed_factor"])
print("Optimizer steps during checks:", gender_e3_check["optimizer_steps"], usage_e3_check["optimizer_steps"])

## 4. Train Gender E3: class-balanced loss

This foreground cell trains folds 0–4. Only the loss changes from the matching Gender E1 parent.

In [ ]:
from fashion.train.task3_experiments import run_task3_child_cv

gender_e3_result = run_task3_child_cv(
    "gender_class_balanced",
    parent_run_ids=gender_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
gender_e3_result

## 5. Train Usage E3: classifier dropout

This foreground cell trains folds 0–4. It keeps the accepted E2 weighted loss and adds only dropout `p=0.20` before the final classifier.

In [ ]:
usage_e3_result = run_task3_child_cv(
    "usage_classifier_dropout",
    parent_run_ids=usage_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
usage_e3_result

## 6. Show the first parent–child comparison

This small table is only a first check. The main Task 3 notebook must still apply every prewritten gate to curves, classes, paired OOF predictions, fold spread, calibration, and robustness.

In [ ]:
import pandas as pd

parent_metric_paths = {
    "gender": DRIVE_TASK_DIR / "baseline/gender/aggregate/metrics.json",
    "usage": DRIVE_TASK_DIR / "experiments/t3_usage_e2_class_balanced_ce/usage/aggregate/metrics.json",
}
children = {"gender": gender_e3_result, "usage": usage_e3_result}
comparison = []
for target, child in children.items():
    parent = json.loads(parent_metric_paths[target].read_text(encoding="utf-8"))
    comparison.append({
        "target": target,
        "parent_macro_f1": parent["macro_f1"],
        "e3_macro_f1": child["metrics"]["macro_f1"],
        "macro_f1_change": child["metrics"]["macro_f1"] - parent["macro_f1"],
        "e3_metrics_path": child["metrics_path"],
    })
pd.DataFrame(comparison)